# BOM Explosion Analysis

Тестовый ноутбук для работы с развертыванием спецификации материалов (Bill of Materials)


In [ ]:
import os
import pandas as pd


# -----------------------
# 0) Load data
# -----------------------
PATH = "../data/task_2_data_ex.xlsx"

df = pd.read_excel(PATH, sheet_name="Лист1")
df.columns = [c.strip().lower() for c in df.columns]


In [500]:
def to_number(x):
    if pd.isna(x):
        return float("nan")
    return float(str(x).replace(",", "").strip())

In [501]:
def add_numeric_qty(df_):
    df_ = df_.copy()
    df_["produced_material_quantity_num"] = df_["produced_material_quantity"].map(to_number)
    df_["component_material_quantity_num"] = df_["component_material_quantity"].map(to_number)
    return df_


In [502]:
def build_edges_year(dy2):
    """
    Yearly BoM edges with yearly component consumption quantity.
    """
    return (
        dy2.groupby(
            [
                "plant_id", "year",
                "produced_material", "produced_material_release_type", "produced_material_production_type",
                "component_material", "component_material_release_type", "component_material_production_type",
            ],
            dropna=False
        )
        .agg(component_consumption_quantity=("component_material_quantity_num", "sum"))
        .reset_index()
    )

In [503]:
def build_mat_year(dy2):
    """
    Yearly produced material quantity:
    max per month (to avoid duplication by components) -> sum over months.
    """
    per_month = (
        dy2.groupby(
            [
                "plant_id", "year", "month",
                "produced_material", "produced_material_release_type", "produced_material_production_type",
            ],
            dropna=False
        )
        .agg(prod_qty_month=("produced_material_quantity_num", "max"))
        .reset_index()
    )

    return (
        per_month.groupby(
            [
                "plant_id", "year",
                "produced_material", "produced_material_release_type", "produced_material_production_type",
            ],
            dropna=False
        )
        .agg(prod_material_production_quantity=("prod_qty_month", "sum"))
        .reset_index()
    )


In [504]:
def build_expandable(mat_year):
    """
    Materials that can be expanded further in explosion (FIN/PROD).
    """
    return set(
        mat_year.loc[mat_year["produced_material_release_type"].isin(["FIN", "PROD"]), "produced_material"]
        .astype(int)
        .unique()
    )

In [505]:
def explode_one_fin(fin_id, bom_map_year, expandable_year):
    rows = []
    stack = [fin_id]
    visited = set()

    while stack:
        parent = stack.pop()
        if parent not in bom_map_year:
            continue

        for c in bom_map_year[parent]:
            edge = (fin_id, parent, c)
            if edge in visited:
                continue
            visited.add(edge)
            rows.append(edge)

            if (c in expandable_year) and (c not in stack):
                stack.append(c)

    return pd.DataFrame(rows, columns=["fin_material_id", "prod_material_id", "component_id"])

In [506]:
def build_edge_lookup(edges_year):
    return {
        (int(r["produced_material"]), int(r["component_material"])): r
        for _, r in edges_year.iterrows()
    }

In [507]:
def build_mat_lookup(mat_year, plant, year):
    m = mat_year[(mat_year["plant_id"] == plant) & (mat_year["year"] == year)]
    return {int(r["produced_material"]): r for _, r in m.iterrows()}


In [508]:
def get_fin_list(mat_year, plant, year):
    m = mat_year[(mat_year["plant_id"] == plant) & (mat_year["year"] == year)]
    return sorted(
        m.loc[m["produced_material_release_type"] == "FIN", "produced_material"]
        .astype(int)
        .unique()
    )

In [509]:
def build_final_for_fin(fin_id, bom_map_year, expandable_year, edge_lookup, mat_lookup, plant, year):
    res_ids = explode_one_fin(fin_id, bom_map_year, expandable_year)

    fin_info = mat_lookup[fin_id]

    rows_out = []
    for _, row in res_ids.iterrows():
        pm = int(row["prod_material_id"])
        cm = int(row["component_id"])

        e = edge_lookup[(pm, cm)]
        pm_info = mat_lookup[pm]

        rows_out.append({
            "plant": plant,
            "fin_material_id": fin_id,
            "fin_material_release_type": fin_info["produced_material_release_type"],
            "fin_material_production_type": fin_info["produced_material_production_type"],
            "fin_production_quantity": fin_info["prod_material_production_quantity"],

            "prod_material_id": pm,
            "prod_material_release_type": pm_info["produced_material_release_type"],
            "prod_material_production_type": pm_info["produced_material_production_type"],
            "prod_material_production_quantity": pm_info["prod_material_production_quantity"],

            "component_id": cm,
            "component_material_release_type": e["component_material_release_type"],
            "component_material_production_type": e["component_material_production_type"],
            "component_consumption_quantity": e["component_consumption_quantity"],

            "year": year
        })

    return pd.DataFrame(rows_out)

In [510]:
def build_final_year(df, plant, year, columns_template=None):
    dy = df[(df["plant_id"] == plant) & (df["year"] == year)].copy()
    if dy.empty:
        return pd.DataFrame(columns=columns_template) if columns_template is not None else pd.DataFrame()

    dy2 = add_numeric_qty(dy)

    edges_year = build_edges_year(dy2)
    mat_year = build_mat_year(dy2)

    bom_map_year = edges_year.groupby("produced_material")["component_material"].apply(list).to_dict()
    expandable_year = build_expandable(mat_year)

    edge_lookup = build_edge_lookup(edges_year)
    mat_lookup = build_mat_lookup(mat_year, plant, year)

    fin_list = get_fin_list(mat_year, plant, year)

    if len(fin_list) == 0:
        return pd.DataFrame(columns=columns_template) if columns_template is not None else pd.DataFrame()

    out = pd.concat(
        [build_final_for_fin(f, bom_map_year, expandable_year, edge_lookup, mat_lookup, plant, year) for f in fin_list],
        ignore_index=True
    )

    if columns_template is not None:
        out = out.reindex(columns=columns_template)

    return out

In [511]:
plants = sorted(df["plant_id"].dropna().unique())
years = [2024, 2025]

all_parts = []
template_cols = None  # чтобы держать один порядок колонок

for plant in plants:
    for year in years:
        part = build_final_year(df, plant, year, columns_template=template_cols)
        if not part.empty:
            all_parts.append(part)
            if template_cols is None:
                template_cols = part.columns  # фиксируем порядок колонок по первому непустому результату

final_all = pd.concat(all_parts, ignore_index=True) if all_parts else pd.DataFrame(columns=template_cols)

final_all.shape



(110, 14)

In [512]:
os.makedirs("../output", exist_ok=True)
final_all.to_csv("../output/bom_explosion_2024_2025.csv", index=False)
final_all.to_excel("../output/bom_explosion_2024_2025.xlsx", index=False)


In [513]:
final_all.shape

(110, 14)

In [514]:
final_all.head(5)

,plant,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_material_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,year
0,RLT_10,10000,FIN,8002,11708.0,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,2024
1,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,80070,PROD,8007.0,11303.0,2024
2,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90000,ADD,NaN,598.0,2024
3,RLT_10,10000,FIN,8002,11708.0,50000,PROD,8002,9538.0,90001,ADD,NaN,242.0,2024
4,RLT_10,10000,FIN,8002,11708.0,80070,PROD,8007,11028.0,80010,PROD,8001.0,41769.0,2024


In [515]:

df.to_csv("../output/bom_raw.csv", index=False)